# Clasificación landuse con ViT preentrenado

Continuación del reto 1: se usa el mismo **ViT-B/16 de KerasHub**
(`vit_base_patch16_224_imagenet`, preentrenado en ImageNet-1k) y el mismo
esquema de fine-tuning en dos fases. No se entrena un ViT desde cero porque el
reto 1 ya demostró que, sin preentrenamiento, el Transformer no supera a la CNN
mejorada (88,75 % frente a 89,17 %; McNemar p = 1,0000) y cuesta mucho más
(2.345 s y 51 épocas). Con preentrenamiento, el mismo ViT sube al 97,50 %
(+8,75 puntos, p = 0,0001) y empata con DenseNet121 (p = 0,6875). El reto 2
parte de ese punto y se concentra en la pregunta que sigue abierta: si el
empate atención/convolución se mantiene en otro dataset.

El ajuste repite el esquema del reto 1:

1. **Fase 1** — backbone congelado; se entrena solo la cabeza `Dropout + Dense`.
2. **Fase 2** — se descongelan los **dos últimos bloques** del encoder y la
   `LayerNormalization` final, y se ajusta con un learning rate bajo.

El ViT divide la imagen en parches de 16×16 y aplica autoatención global. Las
imágenes se cargan a 224×224 y se escalan a `[-1, 1]` dentro del modelo, la
configuración del preset. Los pesos ajustados se guardan en `model.weights.h5`
y están ignorados por git: en un clon nuevo el cuaderno reentrena el modelo.


## Configuración


In [1]:
import keras
import keras_hub
import numpy as np
import pandas as pd
import seaborn as sns
from keras import layers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

import common_landuse as cl

FORCE_RETRAIN = False
PRESET = "vit_base_patch16_224_imagenet"
IMAGE_SIZE = 224
FINE_TUNE_BLOCKS = 2
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 6
BATCH_SIZE = 32
sns.set_theme(style="whitegrid", context="notebook")
keras.utils.set_random_seed(cl.SEED)
np.random.seed(cl.SEED)


I0000 00:00:1795498712.322224    1816 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795498712.357038    1816 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795498713.387381    1816 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/andres/Desktop/vision-por-computador/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carga del dataset y split común


In [2]:
data = cl.load_split_data(size=IMAGE_SIZE)
X_train = data["X_train"].astype("float32")
X_val = data["X_val"].astype("float32")
X_test = data["X_test"].astype("float32")
y_train_labels = data["y_train"]
y_val_labels = data["y_val"]
y_test_labels = data["y_test"]
test_paths = data["test_paths"]

y_train = cl.one_hot(y_train_labels)
y_val = cl.one_hot(y_val_labels)

print("Train:", X_train.shape, "| Validation:", X_val.shape, "| Test:", X_test.shape)
print("Test distribution:", {name: int(np.sum(y_test_labels == name)) for name in cl.CLASSES})
print("Train distribution:", {name: int(np.sum(y_train_labels == name)) for name in cl.CLASSES})


Train: (956, 224, 224, 3) | Validation: (169, 224, 224, 3) | Test: (375, 224, 224, 3)
Test distribution: {'denseresidential': 125, 'harbor': 125, 'river': 125}
Train distribution: {'denseresidential': 319, 'harbor': 319, 'river': 318}


## Modelo, entrenamiento y evaluación


In [3]:
def build_vit():
    inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)  # [0, 255] -> [-1, 1]

    backbone = keras_hub.models.Backbone.from_preset(PRESET)
    backbone.trainable = False

    tokens = backbone(x, training=False)
    class_token = tokens[:, 0]
    x = layers.Dropout(0.2)(class_token)
    outputs = layers.Dense(len(cl.CLASSES), activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="vit_pretrained_landuse")
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    return model


def unfreeze_last_blocks(model, n_blocks=FINE_TUNE_BLOCKS):
    backbone = next(
        layer for layer in model.layers if isinstance(layer, keras_hub.models.Backbone)
    )
    backbone.trainable = True
    backbone.get_layer("vit_patching_and_embedding").trainable = False
    encoder = backbone.get_layer("vit_encoder")
    for block in encoder.encoder_layers[:-n_blocks]:
        block.trainable = False


def fit_vit(model):
    phase1_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]
    history_phase1, seconds_phase1 = cl.train_keras_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=HEAD_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=phase1_callbacks,
    )

    unfreeze_last_blocks(model)
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-5),
        metrics=["accuracy"],
    )
    phase2_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    ]
    history_phase2, seconds_phase2 = cl.train_keras_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=FINE_TUNE_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=phase2_callbacks,
    )

    phase1 = pd.DataFrame(history_phase1.history)
    phase1["stage"] = "head"
    phase2 = pd.DataFrame(history_phase2.history)
    phase2["stage"] = "fine_tune"
    history = pd.concat([phase1, phase2], ignore_index=True)
    history.index = np.arange(1, len(history) + 1)
    return model, history, seconds_phase1 + seconds_phase2


def evaluate_vit(model, history, training_seconds):
    predicted_idx, probabilities, inference_ms = cl.predict_keras(
        model, X_test, batch_size=BATCH_SIZE
    )
    predicted = np.array(cl.CLASSES)[predicted_idx]
    metrics = cl.compute_metrics(
        name="ViT-B/16 preentrenado",
        family="IA generativa (Transformer)",
        y_true=y_test_labels,
        y_pred=predicted,
        y_prob=probabilities,
        params=model.count_params(),
        trainable_params=sum(np.prod(weight.shape) for weight in model.trainable_weights),
        training_seconds=training_seconds,
        inference_ms=inference_ms,
        epochs=len(history),
        hyperparameters={
            "preset": PRESET,
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "head_epochs": HEAD_EPOCHS,
            "fine_tune_epochs": FINE_TUNE_EPOCHS,
            "unfreeze_blocks": FINE_TUNE_BLOCKS,
            "optimizer": "Adam",
            "head_lr": 1e-3,
            "fine_tune_lr": 1e-5,
            "augmentation": "flip + rotation 0.05",
            "rescaling": "1/127.5, offset -1 inside the model",
        },
        notes=(
            "ViT-B/16 pretrained on ImageNet-1k (keras-hub) with two-phase "
            "fine-tuning, aligned with reto 1."
        ),
    )
    cl.save_results(
        "vit",
        metrics,
        y_test_labels,
        predicted,
        probabilities,
        test_paths=test_paths,
        history=history,
        model=None,
    )
    model.save_weights(cl.result_dir("vit") / cl.ARTIFACTS["weights"])
    return metrics


def load_vit(folder="vit"):
    weights_path = cl.result_dir(folder) / cl.ARTIFACTS["weights"]
    if not weights_path.exists():
        return None
    model = build_vit()
    model.load_weights(weights_path)
    return model


def load_or_train_vit():
    return cl.load_or_train(
        "vit",
        build_model=build_vit,
        fit_model=fit_vit,
        evaluate_model=evaluate_vit,
        load_model=load_vit,
        force=FORCE_RETRAIN,
    )


## Ejecución


In [4]:
model, history, metrics, retrained = load_or_train_vit()
print("Test accuracy:", round(metrics["accuracy"], 4))
history.tail()


E0000 00:00:1795498741.766214    1816 cuda_executor.cc:1690] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:68

W0000 00:00:1795498741.766424    2243 cuda_executor.cc:1710] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:77
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:107
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:112

W0000 00:00:1795498741.778398    1816 dlopen_checker.cc:38] Could not load dynamic library 'libcudart.so.12'; dlerror: libcudart.so.12: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/andres/Deskt

[cache] Using trained model from /home/andres/Desktop/vision-por-computador/guia-4/reto-2/resultados/vit
Test accuracy: 0.9947


/home/andres/Desktop/vision-por-computador/venv/lib/python3.14/site-packages/keras/src/saving/saving_lib.py:841: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 74 variables. 
  saveable.load_own_variables(store)


,accuracy,loss,val_accuracy,val_loss,learning_rate,stage
epoch,,,,,,
6,0.997908,0.019971,1.0,0.010725,0.00100,head
7,0.995816,0.031636,1.0,0.017670,0.00001,fine_tune
8,0.997908,0.019355,1.0,0.012604,0.00001,fine_tune
9,0.998954,0.014232,1.0,0.009818,0.00001,fine_tune
10,0.998954,0.013618,1.0,0.007752,0.00001,fine_tune


## Observaciones

* ViT-B/16 aporta **autoatención global**: cada parche puede relacionarse con
  cualquier otro. El preentrenamiento en ImageNet-1k es el factor que lo hace
  competitivo con solo 956 imágenes de entrenamiento.
* La receta es la del reto 1: primero se adapta la cabeza con el backbone
  congelado y después se especializan los dos últimos bloques con un learning
  rate bajo. Así el coste baja frente a entrenar el Transformer desde cero.
* El fine-tuning sigue siendo caro en CPU: ViT-B/16 tiene ~86 M de parámetros y
  trabaja a 224×224. A cambio, necesita menos épocas que un ViT desde cero.
* Los resultados quedan en `resultados/vit/`: métricas, predicciones, historia
  y los pesos en `model.weights.h5`. Ese archivo (~345 MB) está ignorado por
  git; en un clon nuevo se reentrena con `FORCE_RETRAIN = True`.
